# Count the documents of one ifdd-be collection

A demo of a notebook that calls ifdd-be: it asks the backend how many
documents of one collection (default: `products`) are valid now, prints the
number and writes it to `count.json`.

Runs the same way in Kestra (`examples/flows/backend_product_count.yml`) and
on a workstation. Everything that differs between the two comes from
environment variables, the client secret included:

| Variable | Example |
|---|---|
| `IFDD_BE_URL` | `https://rest` |
| `IFDD_OIDC_TOKEN_URL` | `https://id/realms/ifddtest/protocol/openid-connect/token` |
| `IFDD_API_CLIENT_ID` | `ifdd-api-script` (may only read) |
| `IFDD_API_CLIENT_SECRET` | secret of that client |
| `REQUESTS_CA_BUNDLE` | CA file, if the certificates come from a local CA |

The secret is never a parameter: papermill writes parameters into the
executed notebook.

## Parameters

The next cell is tagged `parameters`; papermill replaces its values
(`-p segment functions`).

In [ ]:
segment = "products"  # path segment of the collection in ifdd-be

## Settings from the environment

In [ ]:
import json
import os

import requests

TIMEOUT_SECONDS = 15


def setting(name):
    """The value of an environment variable; stop with a clear message if it is missing."""
    value = os.environ.get(name, "").strip()
    if not value:
        raise SystemExit(f"Environment variable {name} is not set (see the first cell).")
    return value


be_url = setting("IFDD_BE_URL").rstrip("/")
token_url = setting("IFDD_OIDC_TOKEN_URL")
client_id = setting("IFDD_API_CLIENT_ID")
client_secret = setting("IFDD_API_CLIENT_SECRET")
print(f"Backend: {be_url}, client: {client_id}, collection: {segment}")

## Access token

Client credentials: the client asks Keycloak for a token in its own name.

In [ ]:
response = requests.post(
    token_url,
    data={"grant_type": "client_credentials", "client_id": client_id, "client_secret": client_secret},
    timeout=TIMEOUT_SECONDS,
)
if response.status_code != 200:
    raise SystemExit(f"Keycloak refused the token: {response.status_code} {response.text[:200]}")
token = response.json()["access_token"]
print("Token received")

## Count

ifdd-be returns the number of all matches in the header `X-Total-Count`,
so one document per page is enough (`start=0&end=1`).

In [ ]:
response = requests.get(
    f"{be_url}/{segment}",
    params={"start": 0, "end": 1},
    headers={"Authorization": f"Bearer {token}"},
    timeout=TIMEOUT_SECONDS,
)
if response.status_code != 200:
    raise SystemExit(f"ifdd-be answered {response.status_code} for /{segment}: {response.text[:200]}")
count = int(response.headers["X-Total-Count"])
print(f"{count} documents in /{segment}")

## Result

In [ ]:
with open("count.json", "w") as stream:
    json.dump({"segment": segment, "count": count}, stream)
print(json.dumps({"segment": segment, "count": count}))